## 1. Scikit-Learn: The Classical CPU Stack
Scikit-Learn is ideal for CPU-bound microservices and tabular data, but it requires heavy trade-offs when applied to high-dimensional spatial data like images.
- Random Forest (Tree Ensemble)
    - Architecture: An ensemble of decision trees using bagging and feature stochasticity.
    - Trade-offs: It is embarrassingly parallel to train and highly interpretable. However, it flattens the 2D image into a 1D vector, completely destroying the spatial hierarchies of the pixels.  
    - Production Fit: Excellent for low-latency tabular data services. Terrible for high-resolution images because tree depth and RAM usage explode as dimensionality increases.
- Multi-Layer Perceptron (MLP)
    - Architecture: A fully connected, feedforward parametric network.
    - Trade-offs: Can map non-linear relationships, but it also treats inputs as flat vectors. Because every node connects to every pixel, MLPs suffer from "parameter explosion"—requiring massive amounts of memory for relatively shallow networks. They are also highly sensitive to unscaled data. 
    - Production Fit: A fast CPU baseline for simple datasets, but outclassed by modern architectures for any spatial tasks.
- K-Nearest Neighbors (K-NN)
    - Architecture: A non-parametric, distance-based algorithm.
    - Trade-offs: It features "lazy learning" with zero training time. However, its inference complexity is $\mathcal{O}(N \times D)$ (where $N$ is the dataset size and $D$ is the feature dimension).
    - Production Fit: A catastrophic bottleneck for high-throughput APIs. Every single user request requires the server to calculate the distance against all 20,000+ stored examples in real-time.
- Soft Voting Ensemble (MLP + K-NN + Random Forest)
    - Architecture: A meta-estimator that averages the probability arrays of independent agents.
    - Trade-offs: It successfully mitigates individual model weaknesses by combining a parametric model capable of non-linear mapping (MLP), a spatial-memory model (K-NN), and a bagging algorithm that handles feature variance well (Random Forest).
    - Production Fit: In production, you pay the compute and memory penalty of all three models at runtime. The $O(N \times D)$ inference complexity of the K-NN component alone disqualifies this specific ensemble for real-time, high-traffic endpoints compared to the compiled CNN.

## 2. PyTorch & TensorFlow: The Deep Learning Stack
For unstructured data like images, deep learning frameworks are mandatory. Both frameworks implemented a Convolutional Neural Network (CNN).
- CNN Architecture Analysis
    - Unlike Random Forests or MLPs, CNNs possess translational invariance—they can recognize a curve or an edge regardless of where it appears in the image.
    - They use a sliding filter window (weight sharing), which drastically reduces the parameter count compared to a dense MLP while capturing local spatial features.

You can explore exactly how the CNN handles data differently than an MLP here: 

In [1]:
#@title CNN vs. MLP Architecture Comparison
# Load Libraries
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.animation as animation
from IPython.display import HTML
import numpy as np

# Set up canvas figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), facecolor="#0f172a")

# CNN Grid (7x7 Spatial Input)
grid_size = 7
kernel_size = 3
grid_data = np.zeros((grid_size, grid_size))
# Draw a simple '7' pattern on the grid
grid_data[1, 1:6] = 1.0
grid_data[2, 5] = 1.0
grid_data[3, 4] = 1.0
grid_data[4, 3] = 1.0
grid_data[5, 3] = 1.0

# Sliding Steps for Convolution Filter
steps = []
for r in range(grid_size - kernel_size + 1):
    for c in range(grid_size - kernel_size + 1):
        steps.append((r, c))

# Draw Static MLP Nodes (Right Axis)
def draw_mlp_static():
    ax2.set_facecolor("#0f172a")
    ax2.set_title("MLP (Flattened Vector -> Fully Connected)", color="white", fontsize=14, pad=15)
    ax2.axis("off")
    ax2.set_xlim(-0.2, 1.2)
    ax2.set_ylim(-0.1, 1.1)

    # Input layer nodes (representing flattened 49 pixels)
    n_inputs = 10  # Visual representative subset
    n_hidden = 6
    
    input_y = np.linspace(0.05, 0.95, n_inputs)
    hidden_y = np.linspace(0.2, 0.8, n_hidden)
    
    # Draw Connections
    for iy in input_y:
        for hy in hidden_y:
            ax2.plot([0.1, 0.9], [iy, hy], color="#334155", alpha=0.3, lw=1)

    # Draw Input Nodes
    for iy in input_y:
        ax2.scatter(0.1, iy, color="#38bdf8", s=120, zorder=3)
    # Draw Hidden Layer Nodes
    for hy in hidden_y:
        ax2.scatter(0.9, hy, color="#818cf8", s=180, zorder=3)

    ax2.text(
        0.1, -0.05, 
        "Flattened Input\n(49 Nodes)", 
        color="#38bdf8", 
        ha="center", 
        fontsize=10
    )
    ax2.text(
        0.9, -0.05, 
        "Dense Layer\n(Fully Connected)", 
        color="#818cf8", 
        ha="center", 
        fontsize=10
    )

draw_mlp_static()

# Animation Update Function
def update(frame):
    ax1.clear()
    ax1.set_facecolor("#0f172a")
    ax1.set_title("CNN (3x3 Spatial Convolution)", color="white", fontsize=14, pad=15)
    ax1.axis("off")
    
    # Display the 7x7 Grid
    ax1.imshow(grid_data, cmap="gray_r", extent=[0, grid_size, grid_size, 0], alpha=0.8)
    
    # Draw Grid Lines
    for i in range(grid_size + 1):
        ax1.plot([0, grid_size], [i, i], color="#334155", lw=1)
        ax1.plot([i, i], [0, grid_size], color="#334155", lw=1)

    # Get current filter position
    r, c = steps[frame % len(steps)]

    # Draw Sliding Filter Window (3x3 Kernel)
    rect = patches.Rectangle(
        (c, r), kernel_size, kernel_size,
        linewidth=2.5, edgecolor="#38bdf8", facecolor="#0284c7", alpha=0.45
    )
    ax1.add_patch(rect)
    
    # Text Annotation
    ax1.text(
        c + kernel_size / 2, r + kernel_size + 0.25,
        f"Filter Step: {frame + 1} / {len(steps)}\nPreserve Spatial Relationships",
        transform=ax1.transData,
        color="#38bdf8",
        ha="center",
        va="top",
        fontsize=11,
    )

# Create Animation Loop
anim = animation.FuncAnimation(
    fig, update, frames=len(steps), interval=350, repeat=True
)

plt.close(fig)  # Prevent rendering static duplicate frame

# Render interactive HTML5 video in Notebook
HTML(anim.to_html5_video())

## Framework Engineering Trade-offs
While the CNN math is identical across both libraries, the choice of framework dictates your deployment strategy:
- PyTorch: Built around dynamic computational graphs (define-by-run). This makes it highly pythonic and the gold standard for debugging, rapid prototyping, and custom layer architecture. Usually preferred by researcher
- TensorFlow: Built heavily around a robust production ecosystem. While PyTorch is catching up, TensorFlow's integrated suite, including TF Serving for cloud endpoints and TFLite for edge devices, makes it the standard choice for enterprise deployment.